# 13. Modern Random Number Generation: Beginner Guide

### 🌟 What is Modern Random Number Generation in NumPy?
NumPy's modern random generator (`np.random.default_rng()`) uses state-of-the-art bit generators to sample from statistical distributions (Normal, Uniform, Binomial, Poisson) and power probabilistic Monte Carlo simulations.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Modern Generator API (`np.random.default_rng`)**: Replacing legacy global seeds with thread-safe generator instances.
- **Statistical Sampling**: Covers `rng.normal()`, `rng.uniform()`, `rng.integers()`, `rng.choice()`, and `rng.binomial()`.
- **Monte Carlo Simulations**: Vectorized Geometric Brownian Motion and Value at Risk (VaR).


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### 🔹 Initializing Generator: `np.random.default_rng(seed=42)`
Creates explicit modern PCG64 generator. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** Generators calculate values lazily on demand. This lets you process millions of rows without loading the entire dataset into memory at once.

**Syntax:** `rng = np.random.default_rng(seed=42)`


In [2]:
rng = np.random.default_rng(seed=42)
print('Modern BitGenerator Instance:', rng)

Modern BitGenerator Instance: Generator(PCG64)


### 🔹 Normal Distribution: `rng.normal()`
Simulates normally distributed transaction noise. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `rng.normal(loc=amounts.mean(), scale=amounts.std(), size=5)`


In [3]:
sim_amounts = rng.normal(loc=amounts.mean(), scale=amounts.std(), size=5)
print('Simulated Transaction Amounts (Normal):', sim_amounts.round(2))

Simulated Transaction Amounts (Normal): [1181.7   406.97 1438.5  1548.03 -117.92]


### 🔹 Uniform Distribution: `rng.uniform()`
Samples random spending thresholds uniformly. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `rng.uniform(low=amounts.min(), high=amounts.max(), size=5)`


In [4]:
sim_uniform = rng.uniform(low=10.0, high=1000.0, size=5)
print('Uniform Threshold Draws:', sim_uniform.round(2))

Uniform Threshold Draws: [975.87 763.53 788.2  136.83 455.88]


### 🔹 Integer Sampling: `rng.integers()`
Samples random customer IDs and account age offsets. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `rng.integers(1, 1000, size=6)`


In [5]:
sampled_cust_ids = rng.integers(100, 999, size=5)
print('Sampled Random Customer ID Numbers:', sampled_cust_ids)

Sampled Random Customer ID Numbers: [549 433 264 933 802]


### 🔹 Weighted Choice: `rng.choice()`
Samples card types according to transaction market share weights. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `rng.choice(['Visa', 'MasterCard', 'Amex', 'Discover'], size=5, p=[0.5, 0.3, 0.15, 0.05])`


In [6]:
cards = np.array(['Visa', 'MasterCard', 'Amex', 'Discover'])
weights = [0.50, 0.30, 0.15, 0.05]
sim_cards = rng.choice(cards, size=6, p=weights)
print('Weighted Card Type Draws:', sim_cards)

Weighted Card Type Draws: ['Amex' 'Visa' 'Visa' 'MasterCard' 'Visa' 'Amex']


### 🔹 Binomial Distribution: `rng.binomial()`
Simulates fraud occurrence counts across 1000 transaction batches. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `rng.binomial(n=1000, p=fraud_flags.mean(), size=5)`


In [7]:
fraud_prob = float(fraud_flags.mean())
sim_fraud_counts = rng.binomial(n=1000, p=fraud_prob, size=5)
print(f'Simulated Fraud Counts per 1000 Transactions (p={fraud_prob:.3f}):', sim_fraud_counts)

Simulated Fraud Counts per 1000 Transactions (p=0.111): [104  91 108 109 134]


### 🔹 Monte Carlo Portfolio Loss Simulation
Simulates 10,000 portfolio loss trajectories using transaction parameters. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.cumsum(rng.normal(loc=..., scale=..., size=(252, 1000)), axis=0)`


In [8]:
daily_vol = float(amounts.std() / amounts.mean()) / 10
shocks = rng.normal(0, daily_vol, size=(252, 1000))
loss_paths = 100000.0 * np.exp(np.cumsum(shocks, axis=0))
print(f'Simulated 1000 Monte Carlo Paths. Final Mean Value: ${loss_paths[-1].mean():,.2f}')

Simulated 1000 Monte Carlo Paths. Final Mean Value: $155,414.89


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Value at Risk (VaR) Estimation on Simulated Portfolios

**Approach:** Calculate 95% 1-Year Value at Risk (VaR) from simulated terminal portfolio values.
**Syntax:** `-np.percentile((loss_paths[-1] - 100000) / 100000, 5) * 100`


In [9]:
terminal_ret = (loss_paths[-1] - 100000.0) / 100000.0
var_95 = -np.percentile(terminal_ret, 5) * 100
print(f'Estimated 95% Value at Risk (VaR): {var_95:.2f}%')

Estimated 95% Value at Risk (VaR): 76.95%
